# 03 — Debug Training (10k Subset)

## Purpose
Before committing to a full GPU training run, we verify the entire pipeline
on a small 10,000-example subset. This catches shape errors, gradient issues,
and configuration bugs cheaply.

## Architecture Overview
- **Encoder**: 2-layer bidirectional LSTM (embedding 256, hidden 512)
- **Decoder**: 2-layer unidirectional LSTM (embedding 256, hidden 512)
- **Attention**: Luong general (learned bilinear scoring)
- **Teacher forcing**: ratio 1.0 during training
- **Loss**: CrossEntropyLoss with `ignore_index=PAD_ID`

## What we check
1. Data loads correctly
2. Tokenizer encodes/decodes
3. Batches have correct tensor shapes
4. Forward pass succeeds with finite outputs
5. Loss is finite
6. Gradients are finite
7. Optimizer step works
8. Loss decreases over 2–3 epochs

If loss doesn't decrease, we investigate the implementation rather than
immediately changing hyperparameters.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from torch.utils.data import DataLoader, Subset

from configs.config import *
from src.tokenizer.tokenizer_utils import UrduTokenizer
from src.data.dataset import QGenDataset, collate_fn
from src.model.encoder import Encoder
from src.model.decoder import Decoder
from src.model.seq2seq import Seq2Seq
from src.training.train import train_model
from src.training.utils import set_seed, get_device, count_parameters

In [ ]:
set_seed(SEED)
device = get_device()

## Step 1: Load tokenizer and create debug data subsets

In [ ]:
tokenizer = UrduTokenizer(SP_MODEL_PATH)
vocab_size = tokenizer.vocab_size
print(f'Vocab size: {vocab_size}')

train_dataset = QGenDataset(TRAIN_FILE, tokenizer)
valid_dataset = QGenDataset(VALID_FILE, tokenizer)

n_train = min(DEBUG_SUBSET_SIZE, len(train_dataset))
n_valid = min(DEBUG_SUBSET_SIZE // 5, len(valid_dataset))
train_sub = Subset(train_dataset, list(range(n_train)))
valid_sub = Subset(valid_dataset, list(range(n_valid)))

train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_sub, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train subset: {n_train}  Valid subset: {n_valid}')

## Step 2: Verify batch shapes

Expected shapes:
- `src`: `[B, max_src_len]`
- `tgt`: `[B, max_tgt_len]`
- `src_lengths`: `[B]`
- `src_mask`: `[B, max_src_len]`

In [ ]:
batch = next(iter(train_loader))
print('Batch shapes:')
for key, val in batch.items():
    print(f'  {key}: {val.shape}')

## Step 3: Build model and check forward pass

In [ ]:
enc_hidden = HIDDEN_SIZE * 2  # bidirectional
encoder = Encoder(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, PAD_ID)
decoder = Decoder(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, enc_hidden, NUM_LAYERS, DROPOUT, PAD_ID)
model = Seq2Seq(encoder, decoder).to(device)

print(f'Trainable parameters: {count_parameters(model):,}')
print(model)

In [ ]:
# Forward pass check
model.train()
src = batch['src'].to(device)
tgt = batch['tgt'].to(device)
src_lengths = batch['src_lengths']
src_mask = batch['src_mask'].to(device)

outputs, attentions = model(src, src_lengths, src_mask, tgt)
print(f'Output shape: {outputs.shape}  (expected [B, T-1, vocab_size])')
print(f'Attention shape: {attentions.shape}  (expected [B, T-1, S])')
assert torch.isfinite(outputs).all(), 'Non-finite outputs!'
print('✓ Forward pass OK')

## Step 4: Debug training (2–3 epochs)

In [ ]:
config = {
    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'bidirectional': BIDIRECTIONAL,
}

debug_ckpt = CHECKPOINT_DIR / 'debug_model.pt'
logs = train_model(
    model, train_loader, valid_loader, device,
    epochs=DEBUG_EPOCHS, checkpoint_path=debug_ckpt, config=config,
)

In [ ]:
# Verify loss decreased
first = logs[0]['train_loss']
last = logs[-1]['train_loss']
print(f'First epoch loss: {first:.4f}')
print(f'Last epoch loss:  {last:.4f}')
if last < first:
    print('✓ Loss decreased — model is learning')
else:
    print('⚠ Loss did NOT decrease — investigate implementation')

## Step 5: Quick greedy decode test

In [ ]:
from src.decoding.greedy import greedy_decode

model.eval()
# Take a single example
item = train_dataset[0]
src_ids = torch.LongTensor([item['src_ids']]).to(device)
src_len = torch.LongTensor([item['src_length']])
src_mask = (src_ids == PAD_ID)

gen_ids, _ = greedy_decode(model, src_ids, src_len, src_mask)
gen_text = tokenizer.decode(gen_ids)

print(f'Source:    {train_dataset.pairs[0]["source"]}')
print(f'Reference: {train_dataset.pairs[0]["target"]}')
print(f'Generated: {gen_text}')